# 08 — Polar sky map of tan(θ_z) for LSST Deep Drilling Fields

## Objective

Display the trajectory of selected DDFs in **horizontal coordinates** (azimuth, zenith distance)
on a **polar projection** whose geometry matches a compass:

| Direction | Position on plot |
|-----------|------------------|
| North     | top  (θ = 0°)    |
| East      | right (θ = +90°) |
| South     | bottom (θ = 180°) |
| West      | left (θ = −90° / 270°) |

The **radial coordinate** is the zenith distance  $z = 90° - \mathrm{alt}$, so the zenith is at the
centre and the horizon at the outer ring.

Each scatter point is colour-coded by $\tan z$, the quantity that directly scales the DCR dipole
length:
$$l_{\rm dip}(b,z) = \sigma_n(b) \times \tan z \times \frac{180 \times 3600}{\pi}$$

### Scientific goals
1. **Verify** that $\tan z$ (and hence dipole amplitude) is largest near azimuth $\pm 90°$ (East/West),
   where refraction has the largest parallactic component.
2. **Compare** COSMOS vs ECDFS: COSMOS transits near the equator at Cerro Pachón
   ($\phi \approx -30.2°$, $\delta_{\rm COSMOS} \approx +2°$) and therefore reaches much higher
   zenith distances → larger $\tan z$ → more DCR dipoles.

## References
- `06_DDF_parallacticAngle.ipynb` — hour angle & parallactic angle machinery
- `07_DDF_DCR.ipynb` — DCR dipole amplitude formula and $\sigma_n(b)$

---
## 1. Imports

In [ ]:
import os
import warnings

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.colorbar import ColorbarBase

warnings.filterwarnings("ignore")
print("Imports OK")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

---
## 2. Configuration

In [ ]:
# ── I/O paths ─────────────────────────────────────────────────────────────────
NB_TAG = "TOOLS_08_POLARMAP-TANZENITH"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Figures: {os.path.abspath(DIR_FIGS)}")


def savefig(name: str) -> None:
    """Save the current figure as PDF + PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight", dpi=150)
    print(f"  → saved {name}.{{pdf,png}}")

In [ ]:
# ── Rubin/LSST observatory — Cerro Pachón ────────────────────────────────────
RUBIN_LAT_DEG = -30.244728  # φ  (degrees North, negative = southern hemisphere)
RUBIN_LON_DEG = -70.749417  # λ  (degrees East)
RUBIN_LAT_RAD = np.radians(RUBIN_LAT_DEG)

# ── LSST Deep Drilling Fields (RA, Dec) in degrees ───────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}

# ── Fields to plot ────────────────────────────────────────────────────────────
DDF_NAMES = ["COSMOS", "ECDFS", "EDFS"]

# ── Colour cycle for DDF trajectories (used for markers)
DDF_COLORS = {
    "COSMOS": "tab:red",
    "ECDFS": "tab:green",
    "EDFS": "tab:blue",
}

# ── Hour angle grid: -6h to +6h  (in hours, then convert to radians) ─────────
N_HA = 500
HA_H = np.linspace(-6.0, 6.0, N_HA)  # hour angle in hours
HA_DEG = HA_H * 15.0  # 1h = 15°
HA_RAD = np.radians(HA_DEG)

print(f"Observatory latitude : {RUBIN_LAT_DEG:.4f}°")
print(f"Fields to plot       : {DDF_NAMES}")
print(f"Hour angle grid      : {HA_H[0]:.1f}h … {HA_H[-1]:.1f}h  ({N_HA} points)")

---
## 3. Horizontal coordinates from hour angle

Standard spherical-astronomy relations for a source at declination $\delta$
observed from latitude $\phi$ at hour angle $H$:

### Altitude / elevation
$$\sin(\mathrm{alt}) = \sin\phi\,\sin\delta + \cos\phi\,\cos\delta\,\cos H$$

### Zenith distance
$$z = 90° - \mathrm{alt}$$

### Azimuth (measured from North, increasing East)
$$\sin A = \frac{-\cos\delta\,\sin H}{\cos(\mathrm{alt})}$$
$$\cos A = \frac{\sin\delta - \sin\phi\,\sin(\mathrm{alt})}{\cos\phi\,\cos(\mathrm{alt})}$$
$$A = \mathrm{atan2}(\sin A,\, \cos A) \mod 360°$$

> **Convention reminder** — matplotlib polar axes use **mathematical** convention (angle measured
> counter-clockwise from the right = East direction).  To get North-up / East-right we set:
> `ax.set_theta_zero_location('N')` and `ax.set_theta_direction(-1)` (clockwise = East).
> We then pass the astronomical azimuth *as-is* to the plot.

In [ ]:
def altaz_from_ha(dec_deg: float, ha_rad: np.ndarray, lat_rad: float):
    """
    Compute altitude, azimuth, and zenith distance for a source at
    declination *dec_deg* observed at hour angles *ha_rad* from an
    observatory at geodetic latitude *lat_rad*.

    Azimuth convention: measured from North, increasing toward East (0° → N, 90° → E).

    Parameters
    ----------
    dec_deg : float
        Source declination in degrees.
    ha_rad  : array_like
        Hour angle(s) in radians.
    lat_rad : float
        Observatory latitude in radians.

    Returns
    -------
    alt_deg, az_deg, zd_deg : ndarrays
        Altitude, azimuth, and zenith distance in degrees.
    """
    dec_r = np.radians(dec_deg)
    sin_alt = np.sin(lat_rad) * np.sin(dec_r) + np.cos(lat_rad) * np.cos(dec_r) * np.cos(ha_rad)
    alt_r = np.arcsin(np.clip(sin_alt, -1, 1))
    cos_alt = np.cos(alt_r)

    # Azimuth: North=0, East=+90  (astronomical convention)
    sin_az = -np.cos(dec_r) * np.sin(ha_rad) / np.where(cos_alt > 1e-9, cos_alt, 1e-9)
    cos_az = (np.sin(dec_r) - np.sin(lat_rad) * sin_alt) / (
        np.cos(lat_rad) * np.where(cos_alt > 1e-9, cos_alt, 1e-9)
    )
    az_r = np.arctan2(sin_az, cos_az) % (2 * np.pi)

    alt_deg = np.degrees(alt_r)
    az_deg = np.degrees(az_r)
    zd_deg = 90.0 - alt_deg
    return alt_deg, az_deg, zd_deg


# Quick sanity check: COSMOS at transit (H=0)
dec_cos = DEEP_FIELDS["COSMOS"][1]
alt_t, az_t, zd_t = altaz_from_ha(dec_cos, np.array([0.0]), RUBIN_LAT_RAD)
print(f"COSMOS at transit: alt={alt_t[0]:.2f}°  az={az_t[0]:.2f}°  z={zd_t[0]:.2f}°")
# Expected: alt ≈ |φ - δ| from the south ≈ 90 + φ - δ
print(f"  check: 90 + φ - δ = {90 + RUBIN_LAT_DEG - dec_cos:.2f}°  (should match alt above)")

---
## 4. Compute trajectories and tan(z) for each selected DDF

In [ ]:
# Storage dict: name → dict with keys alt, az, zd, tan_z, ha_h
TRAJ = {}

for name in DDF_NAMES:
    ra_deg, dec_deg = DEEP_FIELDS[name]
    alt, az, zd = altaz_from_ha(dec_deg, HA_RAD, RUBIN_LAT_RAD)

    # Filter to observable conditions: alt > 10° to avoid horizon issues
    mask = alt > 10.0
    tan_z = np.tan(np.radians(zd))

    TRAJ[name] = {
        "ha_h": HA_H[mask],
        "alt": alt[mask],
        "az": az[mask],
        "az_rad": np.radians(az[mask]),
        "zd": zd[mask],
        "tan_z": tan_z[mask],
    }
    print(
        f"{name:10s}  dec={dec_deg:+7.3f}°  "
        f"N_obs={mask.sum()}  "
        f"zd_max={zd[mask].max():.1f}°  "
        f"tan_z_max={tan_z[mask].max():.3f}"
    )

---
## 5. Polar sky map — one panel per DDF

### Axis setup
- `set_theta_zero_location('N')` → θ = 0 at top (North)
- `set_theta_direction(-1)` → clockwise angle increase (East to the right, West to the left)
- Radial axis = zenith distance in degrees (0° at centre = zenith, 90° at edge = horizon)
- Dual radial labels: degrees on the outside, airmass $\sec z = 1/\cos z$ as inner annotation

In [ ]:
# ── Shared colour scale across all panels ────────────────────────────────────
all_tan_z = np.concatenate([TRAJ[n]["tan_z"] for n in DDF_NAMES])
VMIN, VMAX = 0.0, np.percentile(all_tan_z, 98)
CMAP = plt.get_cmap("jet")
NORM = mcolors.Normalize(vmin=VMIN, vmax=VMAX)

# ── Radial tick positions (zenith distance in degrees) ────────────────────────
ZD_TICKS_DEG = [0, 15, 30, 45, 60, 70, 80]  # zenith distance tick marks


def setup_polar_ax(ax, title: str = "") -> None:
    """Configure a polar axes for a North-up, East-right sky map."""
    # North at top, East at right (clockwise)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)  # clockwise → East is right

    # Radial range: zenith (0°) to horizon (90°)
    ax.set_rlim(0, 90)

    # Radial ticks with dual labels: degrees + airmass
    ax.set_rticks(ZD_TICKS_DEG)
    r_labels = []
    for zd in ZD_TICKS_DEG:
        if zd == 0:
            r_labels.append("Zenith")
        else:
            sec_z = 1.0 / np.cos(np.radians(zd))
            r_labels.append(f"{zd}° , (X={sec_z:.2f})")
    ax.set_yticklabels(r_labels, fontsize=10, color="grey")

    # Azimuth tick labels: compass points
    az_ticks = np.radians([0, 45, 90, 135, 180, 225, 270, 315])
    az_labels = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    ax.set_thetagrids(np.degrees(az_ticks), labels=az_labels, fontsize=8)

    ax.grid(True, color="grey", alpha=1, linestyle=":")
    if title:
        ax.set_title(title, fontsize=14, fontweight="bold", pad=14)


def add_ha_ticks(ax, traj: dict, n_ticks: int = 13) -> None:
    """
    Overlay small text labels for selected hour-angle values along the trajectory.
    Ticks are placed at integer hour angles: -6h, -5h, …, 0h (transit), …, +6h.
    """
    ha_h = traj["ha_h"]
    az_rad = traj["az_rad"]
    zd = traj["zd"]

    for h_target in range(-6, 7, 1):  # -6h to +6h, integer steps
        idx = np.argmin(np.abs(ha_h - h_target))
        if np.abs(ha_h[idx] - h_target) > 0.2:  # not found within 0.2h tolerance
            continue
        label = f"{h_target:+d}h" if h_target != 0 else "0h"
        ax.annotate(
            label,
            xy=(az_rad[idx], zd[idx]),
            fontsize=10,
            color="black",
            ha="center",
            va="center",
            bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.6),
        )


# ── Figure: one panel per DDF ─────────────────────────────────────────────────
n_fields = len(DDF_NAMES)
fig, axes = plt.subplots(
    1,
    n_fields,
    figsize=(7 * n_fields, 7.5),
    subplot_kw={"projection": "polar"},
)
fig.suptitle(
    r"DDF sky trajectories coloured by $\tan\,\theta_z$  "
    r"($H \in [-6\,\mathrm{h},\,+6\,\mathrm{h}]$, alt $> 10°$)",
    fontsize=12,
    y=1.0,
)

for ax, name in zip(axes, DDF_NAMES):
    traj = TRAJ[name]
    dec = DEEP_FIELDS[name][1]

    setup_polar_ax(ax, title=f"{name}\n($\\delta={dec:+.1f}°$)")

    sc = ax.scatter(
        traj["az_rad"],
        traj["zd"],
        c=traj["tan_z"],
        cmap=CMAP,
        norm=NORM,
        s=30,
        alpha=1.0,
        linewidths=0,
    )
    # Mark the transit (H=0, minimum zenith distance)
    idx_transit = np.argmin(traj["zd"])
    ax.scatter(
        traj["az_rad"][idx_transit],
        traj["zd"][idx_transit],
        marker="*",
        s=120,
        color="gold",
        edgecolors="black",
        linewidths=0.25,
        zorder=5,
        label=f"Transit z={traj['zd'][idx_transit]:.1f}°",
    )
    ax.legend(fontsize=10, loc="lower right", bbox_to_anchor=(1.1, -0.08))
    add_ha_ticks(ax, traj)

# ── Shared colour bar ─────────────────────────────────────────────────────────
cbar_ax = fig.add_axes([0.92, 0.12, 0.015, 0.72])
cb = ColorbarBase(cbar_ax, cmap=CMAP, norm=NORM, orientation="vertical")
cb.set_label(r"$\tan\,\theta_z$", fontsize=11)
cb.ax.tick_params(labelsize=8)

plt.tight_layout(rect=[0, 0, 0.9, 1])
savefig("08_polarmap_tanzenith_individual")
plt.show()

---
## 6. Overlay plot — all DDFs on a single polar panel

All three DDF trajectories are plotted on the same polar axes using the shared `jet` colour map
to encode $\tan z$.  A single, clearly visible colour bar is placed on the right.  Each DDF
trajectory is additionally identified by an **edgecolor** ring around its scatter markers
(red / green / blue for COSMOS / ECDFS / EDFS) so the reader can disentangle the three
trajectories even where they overlap.

In [ ]:
from matplotlib.lines import Line2D

# ── Single shared colour map: jet ─────────────────────────────────────────────
CMAP_OVERLAY = plt.get_cmap("jet")
# NORM is already defined above (shared across all sections)

# ── Figure ────────────────────────────────────────────────────────────────────
fig2 = plt.figure(figsize=(9, 8))

# Polar axes occupies left part; room for a wide colorbar on the right
ax2 = fig2.add_axes([0.05, 0.05, 0.78, 0.88], projection="polar")

# ── Polar axis setup (North-up, East-right) ───────────────────────────────────
ax2.set_theta_zero_location("N")
ax2.set_theta_direction(-1)
ax2.set_rlim(0, 90)

ZD_TICKS_DEG = [0, 15, 30, 45, 60, 70, 80]
ax2.set_rticks(ZD_TICKS_DEG)
r_labels = []
for zd in ZD_TICKS_DEG:
    if zd == 0:
        r_labels.append("Zenith")
    else:
        sec_z = 1.0 / np.cos(np.radians(zd))
        r_labels.append(f"{zd}°  (X={sec_z:.2f})")
ax2.set_yticklabels(r_labels, fontsize=9, color="0.35")
ax2.set_rlabel_position(48)  # put radial labels between N and E to avoid overlap

ax2.set_thetagrids(
    [0, 45, 90, 135, 180, 225, 270, 315],
    labels=["N", "NE", "E", "SE", "S", "SW", "W", "NW"],
    fontsize=12,
)
ax2.grid(True, color="grey", alpha=0.4, linestyle=":")
ax2.set_title(
    r"All selected DDFs — $\tan\,\theta_z$  ($H \in [-6\,\mathrm{h},\,+6\,\mathrm{h}]$, alt $> 10°$)",
    fontsize=12,
    fontweight="bold",
    pad=10,
)

# ── Scatter: one call per DDF, edgecolor encodes field identity ───────────────
for name in DDF_NAMES:
    traj = TRAJ[name]
    dec = DEEP_FIELDS[name][1]
    edge_color = DDF_COLORS[name]  # red / green / blue ring around each marker

    ax2.scatter(
        traj["az_rad"],
        traj["zd"],
        c=traj["tan_z"],
        cmap=CMAP_OVERLAY,
        norm=NORM,
        s=40,
        alpha=1.00,
        linewidths=0,
        edgecolors=edge_color,
        zorder=3,
    )

    # Mark transit with a coloured star
    idx_t = np.argmin(traj["zd"])
    ax2.scatter(
        traj["az_rad"][idx_t],
        traj["zd"][idx_t],
        marker="*",
        s=220,
        # color=edge_color, edgecolors="black", linewidths=0.7, zorder=6,
        color="gold",
        edgecolors="black",
        linewidths=0.7,
        zorder=6,
    )

    # DDF name annotation near transit point
    ax2.annotate(
        name,
        xy=(traj["az_rad"][idx_t], traj["zd"][idx_t]),
        xytext=(traj["az_rad"][idx_t] + 0.18, traj["zd"][idx_t] + 5),
        fontsize=10,
        fontweight="bold",
        color=edge_color,
        arrowprops=dict(arrowstyle="->", color=edge_color, lw=0.9),
        zorder=7,
    )

    # Hour angle labels at integer values
    ha_h = traj["ha_h"]
    az_rad = traj["az_rad"]
    zd_arr = traj["zd"]
    for h_target in range(-6, 7, 1):
        idx = np.argmin(np.abs(ha_h - h_target))
        if np.abs(ha_h[idx] - h_target) > 0.2:
            continue
        lbl = f"{h_target:+d}h" if h_target != 0 else "0h"
        ax2.annotate(
            lbl,
            xy=(az_rad[idx], zd_arr[idx]),
            fontsize=15,
            color=edge_color,
            ha="center",
            va="center",
            bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.15),
            zorder=5,
        )

# ── Legend: field identity (edgecolor key) ────────────────────────────────────
legend_elements = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="0.7",
        markeredgecolor=DDF_COLORS[n],
        markeredgewidth=1.5,
        markersize=10,
        label=f"{n}  (δ = {DEEP_FIELDS[n][1]:+.1f}°,  tan z_max = {TRAJ[n]['tan_z'].max():.2f})",
    )
    for n in DDF_NAMES
]
ax2.legend(
    handles=legend_elements,
    loc="lower left",
    bbox_to_anchor=(-0.05, -0.05),
    fontsize=10,
    title="DDF",
    title_fontsize=9,
    framealpha=0.9,
)

# ── Single, wide, clearly labelled colour bar ─────────────────────────────────
cbar_ax2 = fig2.add_axes([0.87, 0.1, 0.04, 0.78])  # [left, bottom, width, height]
cb2 = ColorbarBase(cbar_ax2, cmap=CMAP_OVERLAY, norm=NORM, orientation="vertical")
cb2.set_label(r"$\tan\,\theta_z$", fontsize=14, labelpad=10)
cb2.ax.tick_params(labelsize=11)
# Add explicit ticks for readability
import matplotlib.ticker as ticker

cb2.locator = ticker.MaxNLocator(nbins=8)
cb2.update_ticks()

savefig("08_polarmap_tanzenith_overlay")
plt.show()

---
## 7. Enhanced version: dual radial labelling in degrees AND hours (airmass)

The radial circles correspond to fixed zenith distances.  We add two annotation layers:
- **outer ring** — zenith distance in degrees (standard)
- **inner annotation** — approximate airmass $X = \sec z$

We also add a secondary annotation showing the **hour angle at which each radial circle is
crossed** for each DDF, to give the observer an intuitive sense of observing windows.

In [ ]:
# ── Compute: for each DDF, at which |H| does the zenith distance equal each tick? ──
ZD_REFERENCE = [30, 45, 60, 70]  # zenith distance reference circles

print("Hour angles at which each DDF crosses reference zenith distances:")
print(f"{'DDF':10s}  {'δ':>8s}  ", end="")
for zd in ZD_REFERENCE:
    print(f" z={zd}°  ", end="")
print()
for name in DDF_NAMES:
    traj = TRAJ[name]
    dec = DEEP_FIELDS[name][1]
    ha_h = traj["ha_h"]
    zd_arr = traj["zd"]
    print(f"{name:10s}  {dec:+8.3f}°  ", end="")
    for zd_ref in ZD_REFERENCE:
        # Find crossing: smallest |H| where zd > zd_ref
        mask_cross = zd_arr > zd_ref
        if mask_cross.any():
            h_cross = np.abs(ha_h[mask_cross]).min()
            print(f" {h_cross:5.2f}h  ", end="")
        else:
            print("   N/A   ", end="")
    print()

In [ ]:
# ── Enhanced polar map with per-DDF single colour map and H annotations ───────
CMAP_SINGLE = plt.get_cmap("jet")

fig3, axes3 = plt.subplots(
    1,
    len(DDF_NAMES),
    figsize=(5 * len(DDF_NAMES), 5.5),
    subplot_kw={"projection": "polar"},
)
fig3.suptitle(
    r"DDF polar sky map — $\tan\,\theta_z$ (shared scale)  "
    r"— Radial labels: zenith distance (°) and airmass $X$",
    fontsize=12,
    y=1.01,
)

for ax, name in zip(axes3, DDF_NAMES):
    traj = TRAJ[name]
    dec = DEEP_FIELDS[name][1]

    # ── polar axis setup ─────────────────────────────────────────────────────
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_rlim(0, 80)

    # Radial ticks: degrees
    zd_ticks = [0, 15, 30, 45, 60, 70]
    ax.set_rticks(zd_ticks)

    # Dual labels: "zd°  (X=...)"  — placed at θ=45° (NE direction) for readability
    r_labels = []
    for zd in zd_ticks:
        if zd == 0:
            r_labels.append("Z")
        else:
            X = 1.0 / np.cos(np.radians(zd))
            r_labels.append(f"{zd}° X={X:.2f}")
    ax.set_yticklabels(r_labels, fontsize=10, color="0.4")
    ax.set_rlabel_position(50)  # put radial labels at az=50° to avoid overlap

    # Azimuth cardinal labels
    ax.set_thetagrids(
        [0, 45, 90, 135, 180, 225, 270, 315],
        labels=["N", "NE", "E", "SE", "S", "SW", "W", "NW"],
        fontsize=16,
    )
    ax.grid(True, color="grey", alpha=0.5, linestyle=":")

    # Title with DDF name and declination
    transit_zd = traj["zd"][np.argmin(traj["zd"])]
    tan_z_max = traj["tan_z"].max()
    ax.set_title(
        f"{name}\n$\\delta = {dec:+.1f}°$   "
        f"$z_{{\\min}} = {transit_zd:.1f}°$   "
        f"$\\tan z_{{\\max}} = {tan_z_max:.2f}$",
        fontsize=14,
        fontweight="bold",
        pad=18,
    )

    # ── scatter ──────────────────────────────────────────────────────────────
    sc = ax.scatter(
        traj["az_rad"],
        traj["zd"],
        c=traj["tan_z"],
        cmap=CMAP_SINGLE,
        norm=NORM,
        s=30,
        alpha=1.0,
        linewidths=0,
        zorder=3,
    )

    # ── transit star ─────────────────────────────────────────────────────────
    idx_t = np.argmin(traj["zd"])
    ax.scatter(
        traj["az_rad"][idx_t],
        traj["zd"][idx_t],
        marker="*",
        s=150,
        color="gold",
        edgecolors="black",
        linewidths=0.6,
        zorder=6,
    )

    # ── Hour angle labels at integer values ───────────────────────────────────
    ha_h = traj["ha_h"]
    az_rad = traj["az_rad"]
    zd_arr = traj["zd"]
    for h_target in range(-6, 7, 1):
        idx = np.argmin(np.abs(ha_h - h_target))
        if np.abs(ha_h[idx] - h_target) > 0.25:
            continue
        lbl = f"{h_target:+d}h" if h_target != 0 else "0h"
        ax.annotate(
            lbl,
            xy=(az_rad[idx], zd_arr[idx]),
            fontsize=10,
            color="navy",
            ha="center",
            va="center",
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.15", fc="lightyellow", ec="lightblue", alpha=0.55, lw=0.1),
            zorder=7,
        )

    # ── Reference zenith-distance circle annotation (H crossing) ─────────────
    for zd_ref, az_ann in zip([45, 60], [np.radians(25), np.radians(25)]):
        mask_cross = zd_arr > zd_ref
        if mask_cross.any():
            h_cross = np.abs(ha_h[mask_cross]).min()
            ax.annotate(
                f"z>{zd_ref}° at |H|>{h_cross:.1f}h",
                xy=(az_ann, zd_ref),
                fontsize=10,
                color="darkred",
                alpha=0.85,
                ha="left",
            )

# ── Shared colour bar ─────────────────────────────────────────────────────────
cbar_ax3 = fig3.add_axes([0.93, 0.12, 0.016, 0.72])
cb3 = ColorbarBase(cbar_ax3, cmap=CMAP_SINGLE, norm=NORM, orientation="vertical")
cb3.set_label(r"$\tan\,\theta_z$", fontsize=12)
cb3.ax.tick_params(labelsize=9)

plt.tight_layout(rect=[0, 0, 0.92, 1])
savefig("08_polarmap_tanzenith_enhanced")
plt.show()

---
## 8. Summary: tan(z) as a function of hour angle  

Supplementary 1-D comparison plot to clearly show that COSMOS reaches much higher
$\tan z$ than ECDFS or EDFS.

In [ ]:
fig4, ax4 = plt.subplots(figsize=(6, 4))

for name in DDF_NAMES:
    traj = TRAJ[name]
    dec = DEEP_FIELDS[name][1]
    color = DDF_COLORS[name]
    ax4.plot(
        traj["ha_h"],
        traj["tan_z"],
        color=color,
        lw=2,
        label=f"{name}  ($\\delta={dec:+.1f}°$)",
    )

ax4.set_xlabel("Hour angle $H$ [h]", fontsize=11)
ax4.set_ylabel(r"$\tan\,\theta_z$", fontsize=11)
ax4.set_title(
    r"$\tan\,\theta_z$ vs hour angle — COSMOS reaches much larger values",
    fontsize=11,
)
ax4.axvline(0, color="grey", lw=0.8, ls="--", label="Transit (H=0)")
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)
ax4.set_xlim(-6, 6)
ax4.set_ylim(0, None)

plt.tight_layout()
savefig("08_tanz_vs_ha")
plt.show()

---
## 9. Conclusions

### What the polar maps show

1. **East–West asymmetry of $\tan z$** — The brightest (highest $\tan z$) portions of each
   trajectory are indeed concentrated at azimuths near $±90°$ (East and West), confirming
   the geometric expectation: when a source rises or sets it transits the East or West and its
   zenith distance is largest.

2. **COSMOS vs ECDFS** — COSMOS ($\delta \approx +2°$) culminates at a zenith distance
   $z_{\min} \approx 90° - |\phi - \delta| \approx 58°$ at Cerro Pachón, while ECDFS
   ($\delta \approx -28°$) passes close to the zenith ($z_{\min} \approx 2°$).
   As a consequence $\tan z_{\max}$(COSMOS) $\gg$ $\tan z_{\max}$(ECDFS), explaining
   why COSMOS produces significantly more — and larger — DCR dipole artifacts.

3. **EDFS** ($\delta \approx -48°$) is always to the South of the zenith at Cerro Pachón,
   reaching moderate zenith distances.  Its $\tan z$ profile sits between COSMOS and ECDFS.

### Implication for dipole analysis
Since $l_{\rm dip} \propto \tan z$, the polar map directly predicts the spatial distribution
of dipole amplitudes across the sky.  Cross-matching with the dipole PA map
(notebook `05_dipole_parallacticcorr.ipynb`) allows a full DCR hypothesis test:
both amplitude *and* orientation must match the theoretical curves.